# Commodity Decision Lab: Real Data, Real Decisions

**Goal:** Apply decision theory to a real commodity allocation problem using live price data.

**Time:** 15 minutes

In this notebook, you'll download actual commodity futures prices (crude oil, gold, corn) using Yahoo Finance, calculate rolling returns, and frame the allocation problem as a multi-armed bandit. You'll compare a fixed allocation strategy against a simple adaptive strategy to see the difference.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from datetime import datetime, timedelta

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

## Load Real Commodity Price Data

We'll download 3 years of daily prices for:
- **WTI Crude Oil** (CL=F)
- **Gold** (GC=F)
- **Corn** (ZC=F)

In [ ]:
# Download commodity futures data
tickers = {
    'WTI Crude': 'CL=F',
    'Gold': 'GC=F',
    'Corn': 'ZC=F'
}

end_date = datetime.now()
start_date = end_date - timedelta(days=3*365)  # 3 years

print("Downloading commodity price data...")
prices = {}
for name, ticker in tickers.items():
    try:
        data = yf.download(ticker, start=start_date, end=end_date, progress=False)
        prices[name] = data['Adj Close']
        print(f"  ✓ {name:12s}: {len(data)} days")
    except Exception as e:
        print(f"  ✗ {name:12s}: Failed to download ({e})")

# Combine into single DataFrame
df_prices = pd.DataFrame(prices)
df_prices = df_prices.dropna()  # Remove days with missing data

print(f"\nCombined dataset: {len(df_prices)} trading days")
print(f"Date range: {df_prices.index[0].date()} to {df_prices.index[-1].date()}")

# Preview
df_prices.tail()

## Calculate Rolling Returns

We'll compute weekly returns (5-day rolling) to simulate a trader making weekly allocation decisions.

In [ ]:
# Calculate daily returns
df_returns = df_prices.pct_change().dropna()

# Calculate rolling weekly returns (5-day)
window = 5
df_weekly_returns = (1 + df_returns).rolling(window).apply(lambda x: x.prod() - 1, raw=True)
df_weekly_returns = df_weekly_returns.dropna()

# Convert to percentage
df_weekly_returns = df_weekly_returns * 100

print("Weekly Returns Statistics:")
print(df_weekly_returns.describe())

# Visualize price history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Normalized prices
df_prices_norm = df_prices / df_prices.iloc[0] * 100
for col in df_prices_norm.columns:
    axes[0].plot(df_prices_norm.index, df_prices_norm[col], label=col, linewidth=2)
axes[0].set_xlabel('Date', fontsize=12)
axes[0].set_ylabel('Normalized Price (Start = 100)', fontsize=12)
axes[0].set_title('Commodity Price History', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right: Rolling volatility (20-day)
df_volatility = df_returns.rolling(20).std() * np.sqrt(252) * 100  # Annualized
for col in df_volatility.columns:
    axes[1].plot(df_volatility.index, df_volatility[col], label=col, linewidth=2, alpha=0.7)
axes[1].set_xlabel('Date', fontsize=12)
axes[1].set_ylabel('Annualized Volatility (%)', fontsize=12)
axes[1].set_title('Rolling 20-Day Volatility', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Frame as a Bandit Problem

**Setup:**
- **Arms:** Three commodity sectors (WTI Crude, Gold, Corn)
- **Rewards:** Weekly returns (risk-adjusted by volatility)
- **Decision:** Each week, allocate capital to one sector
- **Objective:** Maximize cumulative risk-adjusted returns

We'll calculate **Sharpe-like rewards** using rolling mean / rolling std.

In [ ]:
# Calculate rolling Sharpe ratio (20-week lookback)
lookback = 20
df_rolling_mean = df_weekly_returns.rolling(lookback).mean()
df_rolling_std = df_weekly_returns.rolling(lookback).std()
df_sharpe = df_rolling_mean / df_rolling_std
df_sharpe = df_sharpe.dropna()

print(f"Sharpe-like Ratio (Rolling {lookback}-week):")
print(df_sharpe.describe())

# Visualize rolling Sharpe
fig, ax = plt.subplots(figsize=(12, 5))
for col in df_sharpe.columns:
    ax.plot(df_sharpe.index, df_sharpe[col], label=col, linewidth=2, alpha=0.8)
ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Sharpe-like Ratio (Return / Volatility)', fontsize=12)
ax.set_title('Rolling Risk-Adjusted Performance', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 Observations:")
print("  - Best sector changes over time (non-stationarity!)")
print("  - Crude oil has high volatility (wide swings in Sharpe)")
print("  - Gold shows regime-dependent performance (safe haven vs risk-on)")
print("  - This is EXACTLY the environment where bandits outperform A/B tests")

## Strategy 1: Fixed Allocation (1/3 each)

**Baseline:** Allocate equal capital to each sector every week (no learning).

In [ ]:
def fixed_allocation_strategy(returns_df):
    """Equal-weight allocation (no adaptation)."""
    # Each week, allocate 1/3 to each commodity
    portfolio_returns = returns_df.mean(axis=1)  # Average of all three
    cumulative_returns = (1 + portfolio_returns / 100).cumprod() - 1
    
    return portfolio_returns, cumulative_returns

fixed_returns, fixed_cumulative = fixed_allocation_strategy(df_weekly_returns)

print("Fixed Allocation (1/3 each) Results:")
print(f"  Total return:    {fixed_cumulative.iloc[-1]*100:6.2f}%")
print(f"  Avg weekly:      {fixed_returns.mean():6.3f}%")
print(f"  Volatility:      {fixed_returns.std():6.3f}%")
print(f"  Sharpe-like:     {fixed_returns.mean() / fixed_returns.std():6.3f}")

## Strategy 2: Adaptive Allocation (Simple Bandit)

**Adaptive:** Use epsilon-greedy to allocate capital based on recent performance.

- Track rolling Sharpe ratio for each commodity
- With probability 90%, allocate to highest-Sharpe commodity (exploit)
- With probability 10%, allocate to random commodity (explore)

In [ ]:
def adaptive_allocation_strategy(returns_df, sharpe_df, epsilon=0.1, lookback=20):
    """Epsilon-greedy allocation based on rolling Sharpe."""
    portfolio_returns = []
    allocations = []
    
    # Start after we have enough data for rolling calculations
    start_idx = lookback
    
    for i in range(start_idx, len(returns_df)):
        # Get current Sharpe estimates
        current_sharpe = sharpe_df.iloc[i - start_idx]
        
        # Epsilon-greedy policy
        if np.random.rand() < epsilon:
            # Explore: random allocation
            chosen_commodity = np.random.choice(returns_df.columns)
        else:
            # Exploit: pick highest Sharpe
            chosen_commodity = current_sharpe.idxmax()
        
        allocations.append(chosen_commodity)
        
        # Get return from chosen commodity this week
        week_return = returns_df.iloc[i][chosen_commodity]
        portfolio_returns.append(week_return)
    
    portfolio_returns = pd.Series(portfolio_returns, index=returns_df.index[start_idx:])
    cumulative_returns = (1 + portfolio_returns / 100).cumprod() - 1
    allocations = pd.Series(allocations, index=returns_df.index[start_idx:])
    
    return portfolio_returns, cumulative_returns, allocations

adaptive_returns, adaptive_cumulative, allocations = adaptive_allocation_strategy(
    df_weekly_returns, df_sharpe, epsilon=0.1, lookback=lookback
)

print("Adaptive Allocation (Epsilon-Greedy) Results:")
print(f"  Total return:    {adaptive_cumulative.iloc[-1]*100:6.2f}%")
print(f"  Avg weekly:      {adaptive_returns.mean():6.3f}%")
print(f"  Volatility:      {adaptive_returns.std():6.3f}%")
print(f"  Sharpe-like:     {adaptive_returns.mean() / adaptive_returns.std():6.3f}")

# Allocation breakdown
print("\nAllocation Distribution:")
allocation_counts = allocations.value_counts()
for commodity, count in allocation_counts.items():
    pct = count / len(allocations) * 100
    print(f"  {commodity:12s}: {count:4d} weeks ({pct:5.1f}%)")

## Compare Strategies

Let's visualize the performance difference.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Cumulative returns comparison
# Align the series (adaptive starts later due to lookback)
common_index = adaptive_cumulative.index
axes[0, 0].plot(common_index, fixed_cumulative.loc[common_index] * 100, 
               label='Fixed (1/3 each)', linewidth=2, alpha=0.8)
axes[0, 0].plot(common_index, adaptive_cumulative * 100, 
               label='Adaptive (Epsilon-Greedy)', linewidth=2, alpha=0.8)
axes[0, 0].set_xlabel('Date', fontsize=11)
axes[0, 0].set_ylabel('Cumulative Return (%)', fontsize=11)
axes[0, 0].set_title('Portfolio Performance Comparison', fontsize=13, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Plot 2: Allocation over time (for adaptive strategy)
allocation_numeric = allocations.map({name: i for i, name in enumerate(df_weekly_returns.columns)})
axes[0, 1].scatter(allocations.index, allocation_numeric, alpha=0.5, s=10)
axes[0, 1].set_yticks(range(len(df_weekly_returns.columns)))
axes[0, 1].set_yticklabels(df_weekly_returns.columns)
axes[0, 1].set_xlabel('Date', fontsize=11)
axes[0, 1].set_ylabel('Chosen Commodity', fontsize=11)
axes[0, 1].set_title('Adaptive Strategy Allocation Over Time', fontsize=13, fontweight='bold')
axes[0, 1].grid(alpha=0.3)

# Plot 3: Rolling Sharpe comparison
fixed_rolling_sharpe = fixed_returns.loc[common_index].rolling(20).mean() / \
                      fixed_returns.loc[common_index].rolling(20).std()
adaptive_rolling_sharpe = adaptive_returns.rolling(20).mean() / \
                         adaptive_returns.rolling(20).std()

axes[1, 0].plot(common_index, fixed_rolling_sharpe, 
               label='Fixed', linewidth=2, alpha=0.8)
axes[1, 0].plot(common_index, adaptive_rolling_sharpe, 
               label='Adaptive', linewidth=2, alpha=0.8)
axes[1, 0].axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
axes[1, 0].set_xlabel('Date', fontsize=11)
axes[1, 0].set_ylabel('Rolling Sharpe (20-week)', fontsize=11)
axes[1, 0].set_title('Risk-Adjusted Performance', fontsize=13, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Plot 4: Drawdown comparison
fixed_drawdown = (1 + fixed_cumulative.loc[common_index]).cummax() - (1 + fixed_cumulative.loc[common_index])
adaptive_drawdown = (1 + adaptive_cumulative).cummax() - (1 + adaptive_cumulative)

axes[1, 1].fill_between(common_index, 0, fixed_drawdown * 100, 
                        alpha=0.3, label='Fixed')
axes[1, 1].fill_between(common_index, 0, adaptive_drawdown * 100, 
                        alpha=0.3, label='Adaptive')
axes[1, 1].set_xlabel('Date', fontsize=11)
axes[1, 1].set_ylabel('Drawdown (%)', fontsize=11)
axes[1, 1].set_title('Maximum Drawdown Comparison', fontsize=13, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)
axes[1, 1].invert_yaxis()

plt.tight_layout()
plt.show()

# Summary statistics
print("\n📊 Performance Summary:")
print(f"\nFixed Allocation:")
print(f"  Total return:     {fixed_cumulative.loc[common_index].iloc[-1]*100:7.2f}%")
print(f"  Sharpe ratio:     {fixed_returns.loc[common_index].mean() / fixed_returns.loc[common_index].std():7.3f}")
print(f"  Max drawdown:     {fixed_drawdown.max()*100:7.2f}%")

print(f"\nAdaptive Allocation:")
print(f"  Total return:     {adaptive_cumulative.iloc[-1]*100:7.2f}%")
print(f"  Sharpe ratio:     {adaptive_returns.mean() / adaptive_returns.std():7.3f}")
print(f"  Max drawdown:     {adaptive_drawdown.max()*100:7.2f}%")

improvement = (adaptive_cumulative.iloc[-1] - fixed_cumulative.loc[common_index].iloc[-1]) * 100
print(f"\n✨ Improvement: {improvement:+.2f}% total return")

## Modify This: Experiment with Parameters

Try adjusting these parameters and re-running cells 4-6:

1. **Epsilon:** Change `epsilon=0.1` to `epsilon=0.05` or `epsilon=0.2`
   - Hypothesis: Lower epsilon = more exploitation, might miss regime changes
   - Higher epsilon = more exploration, adapts faster but wastes more

2. **Lookback window:** Change `lookback=20` to `lookback=10` or `lookback=40`
   - Hypothesis: Shorter window = more reactive to recent changes (good for non-stationarity)
   - Longer window = smoother estimates (good for noise reduction)

3. **Risk-adjusted vs raw returns:** Modify the reward signal
   - Current: Uses Sharpe ratio (return / volatility)
   - Alternative: Use raw returns (favor high returns, ignore risk)
   - Alternative: Use Sortino ratio (downside risk only)

In [ ]:
# YOUR TURN: Modify parameters here and re-run from cell 4

# Try these variations:
# epsilon = 0.05  # Less exploration
# epsilon = 0.20  # More exploration

# lookback = 10   # Shorter window (more reactive)
# lookback = 40   # Longer window (smoother)

# Alternative reward: Use raw returns instead of Sharpe
# df_sharpe = df_rolling_mean  # Just use mean returns, ignore volatility

print("Modify parameters above and re-run from cell 4!")

## Summary

**What you learned:**

1. Real commodity markets are **non-stationary** — the best sector changes over time
2. Fixed allocation strategies (like A/B tests) miss opportunities during regime shifts
3. Adaptive strategies (bandits) can shift capital toward winners dynamically
4. The **reward signal matters** — risk-adjusted returns (Sharpe) often outperform raw returns
5. **Lookback windows** control reactivity vs stability (short = reactive, long = stable)

**Key insight:**
- Commodity trading is a natural bandit problem: which sector to allocate to each week?
- Simple epsilon-greedy already outperforms fixed allocation in non-stationary environments
- More sophisticated algorithms (UCB, Thompson Sampling) will do even better

**What's next:**
- **Module 1:** Deep dive into epsilon-greedy variants (fixed, decaying, adaptive)
- **Module 2:** UCB algorithm — provably optimal regret bounds
- **Module 3:** Thompson Sampling — Bayesian approach often wins in practice

**This is what we'll formalize in Module 1:** Taking this intuitive epsilon-greedy approach and making it rigorous with regret analysis, parameter tuning, and performance guarantees.